# Day 14 · One-Stage Object Detectors

**Objective:** Implement the Non-Maximum Suppression (NMS) sorting loop from scratch using mock detected-weed bounding boxes.

## 1. Create mock weed detections

Each detection contains a bounding box `[x1, y1, x2, y2]` and a confidence score. Several boxes overlap because a detector may produce multiple predictions for the same weed.

In [ ]:
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches

boxes = torch.tensor([
    [40, 40, 160, 170],
    [48, 45, 158, 168],
    [55, 52, 150, 165],
    [210, 55, 320, 180],
    [218, 62, 318, 178],
    [370, 80, 470, 190]
], dtype=torch.float32)

scores = torch.tensor([0.96, 0.88, 0.72, 0.91, 0.67, 0.83])

## 2. Implement IoU

NMS uses IoU to decide whether two predicted boxes overlap too much.

In [ ]:
def box_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = max(0, box1[2] - box1[0]) * max(0, box1[3] - box1[1])
    area2 = max(0, box2[2] - box2[0]) * max(0, box2[3] - box2[1])
    union = area1 + area2 - intersection
    return intersection / (union + 1e-7)

## 3. Implement Non-Maximum Suppression

NMS repeatedly keeps the highest-confidence box and removes remaining boxes whose IoU with it exceeds the threshold.

In [ ]:
def nms(boxes, scores, iou_threshold=0.5):
    order = torch.argsort(scores, descending=True)
    kept_indices = []

    while len(order) > 0:
        best = order[0].item()
        kept_indices.append(best)

        remaining = order[1:]
        filtered = []

        for idx in remaining:
            overlap = box_iou(boxes[best].tolist(), boxes[idx].tolist())
            if overlap <= iou_threshold:
                filtered.append(idx)

        order = torch.tensor(filtered, dtype=torch.long)

    return kept_indices

kept = nms(boxes, scores, iou_threshold=0.5)
print("Kept detection indices:", kept)
print("Kept confidence scores:", scores[kept].tolist())

## 4. Visualize detections before and after NMS

In [ ]:
def draw_boxes(ax, selected_indices, title):
    for idx in selected_indices:
        x1, y1, x2, y2 = boxes[idx].tolist()
        ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                       linewidth=2, edgecolor="red", facecolor="none"))
        ax.text(x1, y1-5, f"{scores[idx]:.2f}", color="white",
                bbox={"facecolor": "red", "alpha": 0.7})
    ax.set_xlim(0, 520)
    ax.set_ylim(230, 0)
    ax.set_title(title)
    ax.set_aspect("equal")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
draw_boxes(axes[0], range(len(boxes)), "Before NMS")
draw_boxes(axes[1], kept, "After NMS")
plt.tight_layout()
plt.show()

### Day 14 · Key takeaway

NMS removes duplicate overlapping detections by keeping the highest-confidence box and suppressing lower-confidence boxes with IoU above the chosen threshold.